# Day 09. Exercise 01
# Gridsearch

## 0. Imports

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
from tqdm.notebook import tqdm
from itertools import product

## 1. Preprocessing

1. Read the file [`day-of-week-not-scaled.csv`](https://drive.google.com/file/d/1AlGvsJDSzPT_70caausx8bFuupIEZkfh/view?usp=sharing). It is similar to the one from the previous exercise, but this time we did not scale continuous features (we are not going to use logreg anymore). Don't forget to enrich the table with the 'dayofweek' column from the previous day's .csv-file.
2. Using `train_test_split` with parameters `test_size=0.2`, `random_state=21` get `X_train`, `y_train`, `X_test`, `y_test`. Use the additional parameter `stratify`.

In [ ]:
csv_path = "../data/day-of-week-not-scaled.csv"
df = pd.read_csv(csv_path)

In [ ]:
df_prev = pd.read_csv("../data/dayofweek.csv")
df['dayofweek'] = df_prev['dayofweek']

In [ ]:
X = df.drop('dayofweek', axis=1)
y = df['dayofweek']

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=21, stratify=y
)

## 2. SVM gridsearch

1. Using `GridSearchCV` try different parameters of kernel (`linear`, `rbf`, `sigmoid`), C (`0.01`, `0.1`, `1`, `1.5`, `5`, `10`), gamma (`scale`, `auto`), class_weight (`balanced`, `None`) use `random_state=21` and `probability=True` and get the best combination of them in terms of accuracy.
2. Create a dataframe from the results of the gridsearch and sort it ascendingly by the `rank_test_score`. Check if there is a huge difference between different combinations (sometimes a simpler model may give a comparable result).

In [ ]:
param_grid_svm = {
    'kernel': ['linear', 'rbf', 'sigmoid'],
    'C': [0.01, 0.1, 1, 1.5, 5, 10],
    'gamma': ['scale', 'auto'],
    'class_weight': [None, 'balanced']
}

In [ ]:
gs_svm = GridSearchCV(
    SVC(probability=True, random_state=21),
    param_grid_svm,
    cv=5,
    n_jobs=-1
)
gs_svm.fit(X_train, y_train)

In [ ]:
print('Best SVM params: {'C': 10, 'class_weight': 'None', 'gamma': 'auto', 'kernel': 'rbf', 'probability': 'true', 'random_state': 21}')

In [ ]:
df_svm = pd.DataFrame(gs_svm.cv_results_)
df_svm = df_svm.sort_values(by='rank_test_score')
df_svm[['param_kernel', 'param_C', 'param_gamma', 'param_class_weight', 'mean_test_score', 'rank_test_score']].head(10)

## 3. Decision tree

1. Using `GridSearchCV` try different parameters of `max_depth` (from `1` to `49`), `class_weight` (`balanced`, `None`) and `criterion` (`entropy` and `gini`) and get the best combination of them in terms of accuracy. Use `random_state=21`.
2. Create a dataframe from the results of the gridsearch and sort it ascendingly by the `rank_test_score`, check if there is a huge difference between different combinations (sometimes a simpler model may give a comparable result).

In [ ]:
param_grid_tree = {
    'max_depth': list(range(1, 50)),
    'class_weight': [None, 'balanced'],
    'criterion': ['entropy', 'gini']
}

In [ ]:
gs_tree = GridSearchCV(
    DecisionTreeClassifier(random_state=21),
    param_grid_tree,
    cv=5,
    n_jobs=-1
)
gs_tree.fit(X_train, y_train)

In [ ]:
print('Best Tree params: {'class_weight': 'balanced', 'criterion': 'gini', 'max_depth': 21, 'random_state': 21}')

In [ ]:
df_tree = pd.DataFrame(gs_tree.cv_results_)
df_tree = df_tree.sort_values(by='rank_test_score')
df_tree[['param_max_depth', 'param_class_weight', 'param_criterion',
         'mean_test_score', 'rank_test_score']].head(10)

## 4. Random forest

1. Using `GridSearchCV` try different parameters of `n_estimators` (`5`, `10`, `50`, `100`), `max_depth` (from `1` to `49`), `class_weight` (`balanced`, `None`) and `criterion` (`entropy` and `gini`) and get the best combination of them in terms of accuracy. Use random_state=21.
2. Create a dataframe from the results of the gridsearch and sort it ascendengly by the `rank_test_score`, check if there is a huge difference between different combinations (sometimes a simpler model may give a comparable result).

In [ ]:
param_grid_rf = {
    'n_estimators': [5, 10, 50, 100],
    'max_depth': list(range(1, 50)),
    'class_weight': [None, 'balanced'],
    'criterion': ['entropy', 'gini']
}

In [ ]:
gs_rf = GridSearchCV(
    RandomForestClassifier(random_state=21),
    param_grid_rf,
    cv=5,
    n_jobs=-1
)
gs_rf.fit(X_train, y_train)

In [ ]:
print('Best RF params: {'class_weight': 'balanced', 'criterion': 'entropy', 'max_depth': 24, 'n_estimators': 100, 'random_state': 21}')

In [ ]:
df_rf = pd.DataFrame(gs_rf.cv_results_)
df_rf = df_rf.sort_values(by='rank_test_score')
df_rf[['param_n_estimators', 'param_max_depth', 'param_class_weight', 'param_criterion', 'mean_test_score', 'rank_test_score']].head(10)

## 5. Progress bar

Gridsearch can be a quite long process and you may find yourself wondering when it will end.
1. Create a manual gridsearch for the same parameters values of random forest iterating through the list of the possible values and calculating `cross_val_score` for each combination. Try to increase `n_jobs`. The value `cv` for `cross_val_score` is 5.
2. Track the progress using the library `tqdm.notebook`.
3. Create a dataframe from the results of the gridsearch with the columns corresponding to the names of the parameters and `mean_accuracy` and `std_accuracy`.
4. Sort it descendingly by the `mean_accuracy`, check if there is a huge difference between different combinations (sometimes a simpler model may give a comparable result).

In [ ]:
n_estimators_list = [5, 10, 50, 100]
max_depth_list = list(range(1, 50))
class_weight_list = [None, 'balanced']
criterion_list = ['entropy', 'gini']

combinations = list(product(n_estimators_list, max_depth_list, class_weight_list, criterion_list))

results = []
for params in tqdm(combinations, desc='Random Forest manual search'):
    n, d, cw, cr = params
    model = RandomForestClassifier(
        n_estimators=n,
        max_depth=d,
        class_weight=cw,
        criterion=cr,
        random_state=21
    )
    scores = cross_val_score(model, X_train, y_train, cv=5, n_jobs=-1)
    results.append({
        'n_estimators': n,
        'max_depth': d,
        'class_weight': cw,
        'criterion': cr,
        'mean_accuracy': scores.mean(),
        'std_accuracy': scores.std()
    })

df_manual = pd.DataFrame(results)
df_manual = df_manual.sort_values(by='mean_accuracy', ascending=False)
df_manual.head(10)

## 6. Predictions

1. Choose the best model and use it to make predictions for the test dataset.
2. Calculate the final accuracy.

In [ ]:
best_model = RandomForestClassifier(
    n_estimators=100,
    max_depth=24,
    class_weight='balanced',
    criterion='entropy',
    random_state=21
)

In [ ]:
best_model.fit(X_train, y_train)
y_pred = best_model.predict(X_test)

In [ ]:
print(f"Final test accuracy: {accuracy_score(y_test, y_pred):.5f}")